### This is an experiment for new image-text retrieval web ###
NOTES:
- This experiment uses Apple's version of CLIP, namely "DFN5B CLIP ViT H14"
- The following part is only for inference computations. To experiment feature extracting, go to feat-extractor.ipynb

**SETUP**

In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from open_clip import create_model_from_pretrained, get_tokenizer
from pymilvus import MilvusClient
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model, preprocess = create_model_from_pretrained('hf-hub:apple/DFN5B-CLIP-ViT-H-14-384')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
# model.eval()  later
model.to(device)
tokenizer = get_tokenizer('ViT-H-14')

translator_name = 'VietAI/envit5-translation'
translate_tokenizer = AutoTokenizer.from_pretrained(translator_name)
translate_model = AutoModelForSeq2SeqLM.from_pretrained(translator_name)

c:\Users\uwu\AppData\Local\Programs\Python\Python312\Lib\site-packages\open_clip\factory.py:129: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkp

**CONNECT TO CLIENT**

In [2]:
client = MilvusClient(
    uri='https://in03-cbdb10d1d199984.serverless.aws-eu-central-1.cloud.zilliz.com',
    token='6305ff67695e59bf211ca717cb68f74f7367ccfd56195824ba6aad7c07f5924cc6c9da8aadec4354aedc193a997225494903a9d7'
)

**UTILS FUNCTION DEFINING**

In [ ]:
def search(query: str, lang: str, collection_name: str, topk: int, search_params: dict):
    if lang == 'vie':
        output = translate_model.generate(translate_tokenizer(query, return_tensors='pt', padding=True).input_ids.to(device), max_length=1024)
        translated_query = translate_tokenizer.batch_decode(output, skip_special_tokens=True)
        used_query = translated_query[0][3:]
        print(used_query)
    elif lang == 'en':
        used_query = query
    else:
        raise ValueError(f'{lang} is not supported. Supported language are: \'en\' and \'vie\'')
    
    text = tokenizer(used_query, context_length=model.context_length).to(device)
    with torch.no_grad(), torch.amp.autocast(device):
        text_feature = model.encode_text(text)
        text_feature = F.normalize(text_feature, dim=-1)

    text_feature = text_feature.squeeze().tolist()

    search_result = client.search(
        collection_name,
        data=[text_feature],
        limit=topk,
        output_fields=["img_key"],
        search_params=search_params
    )
    
    return search_result

**IMAGE SEARCH**

In [ ]:
query = 'bóng đá'
TOP_K = 100
collection_name = 'aictestbatch'
search_params = {
    'metric_type': 'COSINE',
    'params': {
        'ef': 200
    }
}
results = search(query, 'vie', collection_name, TOP_K, search_params)

**SHOW QUERY RESULT (TEMPORARY SOLUTION)**

In [ ]:
for res in results:
    for hit in res:
        dir = hit['entity']['img_key'].replace('/', '\\')[6:]
        original_path = f'D:\\AIC2024\\kfs\\{dir}'
        print(original_path)

**DEBUGGING**

In [9]:
print('''
    Vietnamese search result:
    D:\AIC2024\kfs\Keyframes_L09\keyframes\L09_V017\\187.jpg
    D:\AIC2024\kfs\Keyframes_L09\keyframes\L09_V017\\205.jpg
    D:\AIC2024\kfs\Keyframes_L09\keyframes\L09_V017\\206.jpg
    D:\AIC2024\kfs\Keyframes_L09\keyframes\L09_V017\\208.jpg
    D:\AIC2024\kfs\Keyframes_L09\keyframes\L09_V017\\207.jpg
    D:\AIC2024\kfs\Keyframes_L09\keyframes\L09_V017\\189.jpg
    D:\AIC2024\kfs\Keyframes_L06\keyframes\L06_V005\\192.jpg
    D:\AIC2024\kfs\Keyframes_L09\keyframes\L09_V017\\203.jpg
    D:\AIC2024\kfs\Keyframes_L09\keyframes\L09_V017\\188.jpg
    D:\AIC2024\kfs\Keyframes_L06\keyframes\L06_V005\\193.jpg
    
    /////////////////////////////////////////////////////////
    
    English search result:
    D:\AIC2024\kfs\Keyframes_L09\keyframes\L09_V017\\187.jpg
    D:\AIC2024\kfs\Keyframes_L09\keyframes\L09_V017\\206.jpg
    D:\AIC2024\kfs\Keyframes_L09\keyframes\L09_V017\\208.jpg
    D:\AIC2024\kfs\Keyframes_L09\keyframes\L09_V017\\205.jpg
    D:\AIC2024\kfs\Keyframes_L09\keyframes\L09_V017\\189.jpg
    D:\AIC2024\kfs\Keyframes_L09\keyframes\L09_V017\\207.jpg
    D:\AIC2024\kfs\Keyframes_L09\keyframes\L09_V017\\188.jpg
    D:\AIC2024\kfs\Keyframes_L09\keyframes\L09_V017\\203.jpg
    D:\AIC2024\kfs\Keyframes_L03\keyframes\L03_V030\\246.jpg
    D:\AIC2024\kfs\Keyframes_L09\keyframes\L09_V017\\202.jpg
''')


    Vietnamese search result:
    D:\AIC2024\kfs\Keyframes_L09\keyframes\L09_V017\187.jpg
    D:\AIC2024\kfs\Keyframes_L09\keyframes\L09_V017\205.jpg
    D:\AIC2024\kfs\Keyframes_L09\keyframes\L09_V017\206.jpg
    D:\AIC2024\kfs\Keyframes_L09\keyframes\L09_V017\208.jpg
    D:\AIC2024\kfs\Keyframes_L09\keyframes\L09_V017\207.jpg
    D:\AIC2024\kfs\Keyframes_L09\keyframes\L09_V017\189.jpg
    D:\AIC2024\kfs\Keyframes_L06\keyframes\L06_V005\192.jpg
    D:\AIC2024\kfs\Keyframes_L09\keyframes\L09_V017\203.jpg
    D:\AIC2024\kfs\Keyframes_L09\keyframes\L09_V017\188.jpg
    D:\AIC2024\kfs\Keyframes_L06\keyframes\L06_V005\193.jpg
    
    /////////////////////////////////////////////////////////
    
    English search result:
    D:\AIC2024\kfs\Keyframes_L09\keyframes\L09_V017\187.jpg
    D:\AIC2024\kfs\Keyframes_L09\keyframes\L09_V017\206.jpg
    D:\AIC2024\kfs\Keyframes_L09\keyframes\L09_V017\208.jpg
    D:\AIC2024\kfs\Keyframes_L09\keyframes\L09_V017\205.jpg
    D:\AIC2024\kfs\Keyframes_L

<>:1: SyntaxWarning: invalid escape sequence '\A'
<>:1: SyntaxWarning: invalid escape sequence '\A'
C:\Users\uwu\AppData\Local\Temp\ipykernel_1360\894815465.py:1: SyntaxWarning: invalid escape sequence '\A'
  print('''


In [11]:
print('''
    Vietnamese search result:
    D:\AIC2024\kfs\Keyframes_L30\keyframes\L30_V059\\028.jpg
    D:\AIC2024\kfs\Keyframes_L30\keyframes\L30_V059\\029.jpg
    D:\AIC2024\kfs\Keyframes_L17\keyframes\L17_V005\\204.jpg
    D:\AIC2024\kfs\Keyframes_L27\keyframes\L27_V016\\194.jpg
    D:\AIC2024\kfs\Keyframes_L28\keyframes\L28_V006\\450.jpg
    D:\AIC2024\kfs\Keyframes_L28\keyframes\L28_V006\\448.jpg
    D:\AIC2024\kfs\Keyframes_L28\keyframes\L28_V006\\422.jpg
    D:\AIC2024\kfs\Keyframes_L28\keyframes\L28_V006\\442.jpg
    D:\AIC2024\kfs\Keyframes_L27\keyframes\L27_V016\\191.jpg
    D:\AIC2024\kfs\Keyframes_L28\keyframes\L28_V006\\424.jpg
      
    //////////////////////////////////////////////////////////////////
      
    English search result:
    D:\AIC2024\kfs\Keyframes_L30\keyframes\L30_V059\\028.jpg
    D:\AIC2024\kfs\Keyframes_L17\keyframes\L17_V005\\204.jpg
    D:\AIC2024\kfs\Keyframes_L17\keyframes\L17_V005\\203.jpg
    D:\AIC2024\kfs\Keyframes_L30\keyframes\L30_V059\\029.jpg
    D:\AIC2024\kfs\Keyframes_L30\keyframes\L30_V013\\012.jpg
    D:\AIC2024\kfs\Keyframes_L28\keyframes\L28_V006\\448.jpg
    D:\AIC2024\kfs\Keyframes_L28\keyframes\L28_V006\\450.jpg
    D:\AIC2024\kfs\Keyframes_L28\keyframes\L28_V006\\442.jpg
    D:\AIC2024\kfs\Keyframes_L28\keyframes\L28_V006\\451.jpg
    D:\AIC2024\kfs\Keyframes_L28\keyframes\L28_V006\\476.jpg
    ''')


    Vietnamese search result:
    D:\AIC2024\kfs\Keyframes_L30\keyframes\L30_V059\028.jpg
    D:\AIC2024\kfs\Keyframes_L30\keyframes\L30_V059\029.jpg
    D:\AIC2024\kfs\Keyframes_L17\keyframes\L17_V005\204.jpg
    D:\AIC2024\kfs\Keyframes_L27\keyframes\L27_V016\194.jpg
    D:\AIC2024\kfs\Keyframes_L28\keyframes\L28_V006\450.jpg
    D:\AIC2024\kfs\Keyframes_L28\keyframes\L28_V006\448.jpg
    D:\AIC2024\kfs\Keyframes_L28\keyframes\L28_V006\422.jpg
    D:\AIC2024\kfs\Keyframes_L28\keyframes\L28_V006\442.jpg
    D:\AIC2024\kfs\Keyframes_L27\keyframes\L27_V016\191.jpg
    D:\AIC2024\kfs\Keyframes_L28\keyframes\L28_V006\424.jpg
      
    //////////////////////////////////////////////////////////////////
      
    English search result:
    D:\AIC2024\kfs\Keyframes_L30\keyframes\L30_V059\028.jpg
    D:\AIC2024\kfs\Keyframes_L17\keyframes\L17_V005\204.jpg
    D:\AIC2024\kfs\Keyframes_L17\keyframes\L17_V005\203.jpg
    D:\AIC2024\kfs\Keyframes_L30\keyframes\L30_V059\029.jpg
    D:\AIC2024\kf

<>:1: SyntaxWarning: invalid escape sequence '\A'
<>:1: SyntaxWarning: invalid escape sequence '\A'
C:\Users\uwu\AppData\Local\Temp\ipykernel_1360\2186978441.py:1: SyntaxWarning: invalid escape sequence '\A'
  print('''
